# แยกส่วนประกอบของสินค้าในภาพเดียว (ตัวอย่าง: Rack server)
สินค้าหนึ่งชิ้นมีหลายส่วน: ช่องไดรฟ์ 8 ช่อง ไฟสถานะ พอร์ต USB ฉลาก  งานตรวจคือ **"แต่ละส่วนครบและถูกต้องไหม"** ไม่ใช่แค่ผ่าน/ไม่ผ่านทั้งเครื่อง

3 วิธี จากง่ายไปซับซ้อน

| วิธี | หลักการ | เหมาะเมื่อ |
|---|---|---|
| **1. แผนที่โซน (ROI map)** | กำหนดกรอบของแต่ละส่วนจากภาพอ้างอิง แล้วตรวจทีละกรอบ | สินค้าอยู่ตำแหน่งเดิมเสมอ (จิ๊ก) ใช้มากที่สุดในโรงงาน |
| **2. หาส่วนประกอบอัตโนมัติ** | หาวัตถุทั้งหมดในภาพ แล้วจำแนกด้วยขนาด รูปร่าง สี | สินค้าเลื่อนได้บ้าง ส่วนประกอบหน้าตาต่างกันชัด |
| **3. AI: Object Detection / Segmentation** | สอนโมเดลให้หาและตีกรอบ/ระบายสีแต่ละส่วน | ส่วนประกอบหน้าตาหลากหลาย หลายรุ่น มุมกล้องไม่คงที่ |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
import cv2
import numpy as np
from vision_camera import show, img_path, put_label
RACK = img_path("rack")
def load(name):
    return cv2.imread(os.path.join(RACK, name))
ref = load("rack_reference.png")
show(ref, titles=["ภาพอ้างอิง: เครื่องที่ครบถ้วน"], size=7)

## 1. แผนที่โซน (ROI map): บอกโปรแกรมว่าแต่ละส่วนอยู่ตรงไหน
วิศวกรกำหนดกรอบของแต่ละส่วนครั้งเดียวจากภาพอ้างอิง (ในโปรแกรมจริงมักลากกรอบด้วยเมาส์)  แล้วแต่ละกรอบมี**กฎตรวจของตัวเอง**:
- ช่องไดรฟ์: ต้องมีไดรฟ์ (พื้นที่สว่างมากพอ)
- ไฟสถานะ: ต้องเป็น**สีเขียว** (ส้ม = fault)
- พอร์ต USB: ต้องมี
- ฉลาก: ต้องมี (พื้นที่ขาวมากพอ)

In [ ]:
ZONES = {}
for i in range(8):
    ZONES[f"bay{i+1}"] = ("drive", (50 + i * 88, 80, 80, 170))        # (x, y, w, h)
ZONES["led_power"]  = ("led",   (48, 28, 24, 24))
ZONES["led_status"] = ("led",   (88, 28, 24, 24))
ZONES["usb1"]       = ("port",  (640, 28, 36, 24))
ZONES["usb2"]       = ("port",  (690, 28, 36, 24))
ZONES["label"]      = ("label", (300, 22, 260, 36))

def check_zone(kind, roi):
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    if kind == "drive":                      # ไดรฟ์ = พื้นที่สว่าง (>90) เกิน 60% ของกรอบ
        return (gray > 90).mean() > 0.6, f"bright {(gray > 90).mean():.0%}"
    if kind == "led":                        # ไฟ = สีเขียว H 40-80  ส้ม H 10-25
        green = cv2.inRange(hsv, (40, 80, 80), (80, 255, 255)).mean() / 255
        orange = cv2.inRange(hsv, (10, 80, 80), (25, 255, 255)).mean() / 255
        if green > 0.3:  return True, "green"
        if orange > 0.3: return False, "ORANGE = fault"
        return False, "off"
    if kind == "port":                       # พอร์ต = มีขอบสว่างล้อมช่องมืด
        return (gray > 150).mean() > 0.25, f"bright {(gray > 150).mean():.0%}"
    if kind == "label":                      # ฉลาก = พื้นที่ขาว > 50%
        return (gray > 200).mean() > 0.5, f"white {(gray > 200).mean():.0%}"

def inspect_zones(img):
    out = img.copy(); report = []
    for name, (kind, (x, y, w, h)) in ZONES.items():
        ok, info = check_zone(kind, img[y:y + h, x:x + w])
        color = (0, 200, 0) if ok else (0, 0, 255)
        cv2.rectangle(out, (x, y), (x + w, y + h), color, 2)
        if not ok:
            put_label(out, name, org=(x, y - 4), scale=0.5, color=(0, 0, 255))
            report.append(f"{name}: {info}")
    return out, report

for name in ["rack_01_good.png", "rack_02_missing_bay.png", "rack_03_led_fault.png", "rack_04_no_port_label.png"]:
    out, report = inspect_zones(load(name))
    print(f"{name:26s} -> " + ("OK" if not report else "NG: " + "; ".join(report)))
    show(out, titles=[name + "  " + ("OK" if not report else "NG")], size=7)

**สังเกต:** ผลไม่ใช่แค่ NG แต่บอกว่า**ส่วนไหน**ผิด (bay6 หาย, led_status เป็นสีส้ม, usb หาย, label หาย) ซึ่งคือสิ่งที่พนักงานหน้าไลน์ต้องการ
**ข้อจำกัด:** ถ้าเครื่องเลื่อนไป 20 พิกเซล กรอบทุกอันจะคลาด  แก้ได้โดยหาตำแหน่งเครื่องก่อน (Template matching จาก notebook 06) แล้วเลื่อนกรอบตาม
**ลองเอง:** เพิ่มโซน `led_power` ให้ต้องเป็นสีเขียวเท่านั้น หรือเปลี่ยนเกณฑ์ไดรฟ์จาก 60% เป็น 90% ดูว่าเกิดอะไรขึ้น

## 2. หาส่วนประกอบอัตโนมัติ: ไม่ต้องกำหนดกรอบ
ให้โปรแกรมหา "ก้อน" ทุกก้อนในภาพเอง แล้ว**จำแนกชนิดจากขนาดและรูปร่าง** (ไดรฟ์ = สี่เหลี่ยมใหญ่ตั้ง, พอร์ต = สี่เหลี่ยมเล็กนอน, ไฟ = วงกลมเล็ก, ฉลาก = สี่เหลี่ยมกว้างสีขาว)
ข้อดี: เครื่องเลื่อนได้ และนับได้ว่ามีไดรฟ์กี่ตัวโดยไม่ต้องรู้ตำแหน่งล่วงหน้า

In [ ]:
def find_components(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, bw = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY)             # ส่วนประกอบสว่างกว่าตัวเครื่อง (ขอบเครื่อง ~90)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    contours, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = img.copy(); found = {"drive": 0, "port": 0, "led": 0, "label": 0}
    for c in contours:
        x, y, w, h = cv2.boundingRect(c); area = w * h
        if area < 150 or (w > 700):                                      # ตัดจุดรบกวน และกรอบตัวเครื่อง
            continue
        aspect = w / h
        mean = gray[y:y + h, x:x + w].mean()
        if area > 8000 and aspect < 0.7:    kind = "drive"
        elif mean > 180 and aspect > 4:     kind = "label"
        elif 500 < area < 1500 and aspect > 1.2: kind = "port"
        elif aspect > 0.7 and area < 500:   kind = "led"
        else:
            continue
        found[kind] += 1
        colors = {"drive": (0, 200, 0), "port": (255, 150, 0), "led": (0, 255, 255), "label": (255, 0, 200)}
        cv2.rectangle(out, (x, y), (x + w, y + h), colors[kind], 2)
        put_label(out, kind, org=(x, y - 4), scale=0.45, color=colors[kind])
    return out, found

EXPECT = {"drive": 8, "port": 2, "led": 2, "label": 1}
for name in ["rack_01_good.png", "rack_02_missing_bay.png", "rack_04_no_port_label.png"]:
    out, found = find_components(load(name))
    missing = {k: EXPECT[k] - v for k, v in found.items() if v < EXPECT[k]}
    print(f"{name:26s} พบ {found}  " + ("OK" if not missing else f"NG ขาด {missing}"))
    show(out, titles=[name], size=7)

**สังเกต:** วิธีนี้นับได้ว่าไดรฟ์ขาด 1 ตัว แม้ไม่รู้ตำแหน่ง แต่กฎจำแนกด้วยขนาด/รูปร่างต้องเขียนใหม่ทุกครั้งที่สินค้าเปลี่ยนรุ่น  และถ้าสองส่วนหน้าตาคล้ายกัน (ไฟสถานะไดรฟ์ กับไฟสถานะเครื่อง) ต้องใช้ตำแหน่งช่วยแยกอยู่ดี

## 3. AI: Object Detection / Segmentation
เมื่อสินค้ามีหลายรุ่น ส่วนประกอบหน้าตาหลากหลาย หรือมุมกล้องไม่คงที่ วิธีเขียนกฎจะไม่ไหว  ใช้ AI ที่**ตีกรอบ** (Detection) หรือ**ระบายสีขอบเขต** (Segmentation) ของแต่ละส่วน

| ต้องการ | เครื่องมือ | สิ่งที่ต้องเตรียม |
|---|---|---|
| ตีกรอบส่วนประกอบ + นับ | **YOLO (detect)** | ภาพจริง + ตีกรอบติดป้ายทุกส่วน (ใช้ Roboflow / LabelImg) หลักร้อยภาพ |
| ขอบเขตละเอียดระดับพิกเซล | **YOLO (segment)** | เหมือนข้างบน แต่ติดป้ายเป็นรูปทรง (ใช้เวลามากกว่า) |
| แบ่งส่วนทันทีโดยไม่ต้องสอน | **SAM (Segment Anything)** | ไม่ต้องสอน แต่ต้องบอกว่าแต่ละส่วนคืออะไรเอง และหนัก |

ตัวอย่างโค้ด YOLO detect (ต้อง `pip install ultralytics` ประมาณ 1 GB):
```python
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.train(data="rack.yaml", epochs=50)       # rack.yaml ชี้ไปโฟลเดอร์ภาพ + ไฟล์กรอบ class: drive, led, usb, label
result = model("rack_front.png")[0]
for box in result.boxes:                       # แต่ละกรอบที่เจอ: ชนิด ตำแหน่ง ความมั่นใจ
    print(result.names[int(box.cls)], box.xyxy[0].tolist(), float(box.conf))
```
จากกรอบที่ได้ ก็นับไดรฟ์ ตรวจสีไฟ อ่านฉลาก ด้วยวิธีจาก notebook 06 ต่อได้ทันที  **AI หาส่วน  กฎง่ายๆ ตรวจส่วน**  คือรูปแบบที่ใช้จริงบ่อยที่สุด

## สรุปสำหรับ Rack server
1. จิ๊กยึดเครื่องได้ รุ่นเดียว → **ROI map** เขียนเสร็จในวันเดียว อธิบายได้ทุกกรอบ
2. เครื่องเลื่อนได้บ้าง รุ่นเดียว → ROI map + Template matching หาตำแหน่งก่อน
3. หลายรุ่น มุมกล้องเปลี่ยน → **YOLO detect** หาส่วนประกอบ แล้วใช้กฎตรวจแต่ละส่วน